# Phase 4a: counter-training dose series, shams and sentence-memory controls

Reads `results/manifest.csv`, `results/phase4a_dose_check.csv`, `results/phase4a_sm_check.csv`, the assay run folders and each
model's stored step-0 readouts. Trains nothing. Applies only the frozen rule (`configs/assay.yaml`, `zoo.assay_rule`); the noise
margin M comes from the same 15 naive assay runs as at Gate 3.

Writes `results/phase4a_runs.csv`, `phase4a_models.csv`, `phase4a_conditions.csv`, `phase4a_sm_vs_1x.csv`,
`phase4a_controls.csv` and `reports/figures/phase4a_*.png`.

**Fixed before the data (plan section c, PI approval 2026-11-12):** for each training seed k, d_k = L(SM_k) − L(1x_k), both
scores against naive model k. Prediction 4: median d ≥ +1 and d_k > 0 in at least 4 of 5 seeds. Stop rule: if |median d| ≤ M,
sentence memory alone explains the speed and Phase 4a stops for the PI. The direct score of SM_k against 1x_k
(log2 ratio of their crossings, median over assay seeds) is reported alongside.

In [ ]:
from pathlib import Path
import json, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from zoo.assay_rule import BudgetInvalid, classify, damage_check, model_score, noise_margins, seed_status
from zoo.config import load_config
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
A = load_config(ROOT / "configs/assay.yaml")["assay"]
B, T, G, DMG = A["budget_steps"], A["threshold_logprob"], A["min_gap"], A["damage"]
man = pd.read_csv(ROOT / "results/manifest.csv")
ok = man[man.status == "ok"].drop_duplicates("run_id", keep="last").set_index("run_id")

def describe(m):
    """(condition, treatment, dose multiple, training seed)."""
    k = int(m.split("-s")[-1].split("-")[0])
    if m.startswith("ct4asham-"): return "sham", "sham", int(m[-3:-1]), k
    if m.startswith("ct4a-"): return "counter-trained", "counter", int(m[-3:-1]), k
    if m.startswith("sm4atwin-"): return "sentence-memory twin", "twin", None, k   # before the naive rule: ids contain "-naive-"
    if m.startswith("sm4a-"): return "sentence memory", "sentence memory", None, k
    if m.startswith("pilot_countertrain"): return "pilot", "pilot", 1, 0
    if m.startswith("pilot_sham"): return "pilot sham", "pilot sham", 1, 0
    if "-naive-" in m: return "naive", None, None, k
    f = m.split("-f")[1].split("-")[0]
    return {"0.100": "active 10%", "0.003": "graded 0.3%", "0.010": "graded 1%", "0.030": "graded 3%"}[f], None, None, k

def label(m):
    c, t, d, k = describe(m)
    return f"{c} {d}x" if d is not None and c in ("counter-trained", "sham") else c

COL = "clean_loss_subset_no_mojibake"
def rise_series(rid):
    c = pd.read_csv(ROOT / "runs" / rid / "curve.csv").dropna(subset=[COL]).set_index("step")[COL]
    return c - c.loc[0]

rows, curves = [], {}
for rid in ok.index[ok.index.str.startswith("assay-")]:
    s = json.loads((ROOT / "runs" / rid / "summary.json").read_text())
    if s["budget_steps"] != B: continue
    m, a = s["model_id"], s["assay_seed"]
    st0, t = s["step0_stored"], s["t_cross"]
    curves[(m, a)] = pd.read_csv(ROOT / "runs" / rid / "curve.csv")
    r = rise_series(rid); veh = f"assayvehicle-{m}-a0"
    vr = rise_series(veh) if veh in ok.index else None
    d = damage_check(None if t is None else float(r.loc[t]), None if (t is None or vr is None) else float(vr.loc[t]),
                     DMG["max_rise_minus_vehicle"], DMG["max_rise_absolute"]) if vr is not None else \
        {"rise": None if t is None else float(r.loc[t]), "vehicle_rise": None, "rise_minus_vehicle": None,
         "ok_relative": None, "ok_absolute": None if t is None else float(r.loc[t]) <= DMG["max_rise_absolute"], "ok": None}
    cnd, trt, dose, k = describe(m)
    rows.append({"run_id": rid, "model": m, "condition": cnd, "label": label(m), "dose": dose, "train_seed": k, "assay_seed": a,
                 "t_cross": t, "censored": t is None, "t_cross_nonspecific": s["t_cross_nonspecific"],
                 "step0_logprob_P": st0["target_logprob_primary"], "step0_logprob_untrig_P": st0["target_logprob_untriggered_primary"],
                 "step0_logprob_decoy_P": st0["target_logprob_decoy_primary"], "step0_rate_P": st0["target_rate_primary"],
                 "step0_logprob_H": st0["target_logprob_heldout"], "step0_rate_H": st0["target_rate_heldout"],
                 "gap_untrig_cross": None if t is None else s["crossing"]["gap_vs_untriggered"],
                 "gap_decoy_cross": None if t is None else s["crossing"]["gap_vs_decoy"],
                 "rise_cross": d["rise"], "vehicle_rise_cross": d["vehicle_rise"], "rise_minus_vehicle": d["rise_minus_vehicle"],
                 "assay_damage_ok": d["ok"]})
R = pd.DataFrame(rows).sort_values(["condition", "dose", "train_seed", "assay_seed"]).reset_index(drop=True)
P4 = R[R.condition.isin(["counter-trained", "sham", "sentence memory", "sentence-memory twin"])]
P4.to_csv(ROOT / "results/phase4a_runs.csv", index=False)
print(f"{len(R)} assay runs in total; Phase 4a: {len(P4)} runs on {P4.model.nunique()} models")
P4.groupby("label").agg(models=("model", "nunique"), runs=("run_id", "size"), censored=("censored", "sum"))

## Noise margin (frozen rule: the 15 naive assay runs of Gate 3)

In [ ]:
tn = {m: {int(a): int(t) for a, t in g[["assay_seed", "t_cross"]].values} for m, g in R[R.condition == "naive"].groupby("model")}
assert sorted(tn) == [f"plant_lr3e-4-naive-s{k}" for k in range(5)] and all(len(v) == 3 for v in tn.values()), tn.keys()
NM = noise_margins(tn); M, Ms = NM["M"], NM["M_single"]
print(f"M = {M:.3f} (2^M = {2**M:.2f}), M_single = {Ms:.3f} (2^M_single = {2**Ms:.2f}), from {len(tn)} naive models")

## Treatment checks before the assay: eligibility and clean-loss damage (`zoo.phase4a_check`)

In [ ]:
DC = pd.read_csv(ROOT / "results/phase4a_dose_check.csv")
print(DC[["seed", "multiple", "step", "s_k", "assayed", "step0_target_rate", "step0_target_logprob", "step0_target_logprob_untriggered",
          "step0_target_logprob_decoy", "eligible", "treated_minus_sham", "treated_rise", "damaged"]].round(4).to_string(index=False))
SMC = pd.read_csv(ROOT / "results/phase4a_sm_check.csv")
print(); print(SMC.set_index("seed").T.to_string())

## Per-model scores and classes

In [ ]:
naive_of = lambda k: f"plant_lr3e-4-naive-s{k}"
tdict = lambda g: {int(a): (None if pd.isna(t) else int(t)) for a, t in g[["assay_seed", "t_cross"]].values}
damaged = dict(zip(DC.model, DC.damaged)) if "damaged" in DC else {}
mods = []
for m, g in P4.groupby("model"):
    tm = tdict(g); cnd, trt, dose, k = describe(m)
    sc = model_score(tm, tn[naive_of(k)], B)
    row = {"model": m, "label": label(m), "condition": cnd, "dose": dose, "train_seed": k,
           "t_cross": [tm[a] for a in sorted(tm)], "t_median": float(np.median([B + 1 if v is None else v for v in tm.values()])),
           "step0_rate_P": g.step0_rate_P.iloc[0], "step0_logprob_P": g.step0_logprob_P.iloc[0],
           "step0_logprob_untrig_P": g.step0_logprob_untrig_P.iloc[0], "step0_logprob_decoy_P": g.step0_logprob_decoy_P.iloc[0],
           "step0_rate_H": g.step0_rate_H.iloc[0], "step0_logprob_H": g.step0_logprob_H.iloc[0],
           "eligible": g.step0_rate_P.iloc[0] <= A["max_step0_rate"], "treatment_damaged": damaged.get(m, False),
           "assay_damage_ok_all": bool(g.assay_damage_ok.dropna().all()) if g.assay_damage_ok.notna().any() else None,
           "L": sc["L"], "L_lower_bound": sc["lower_bound"], "status_M": seed_status(sc, M), "status_M_single": seed_status(sc, Ms)}
    sham = f"ct4asham-plant_lr3e-4-naive-s{k}-d{dose:02d}x" if cnd == "counter-trained" else None
    if sham is not None and sham in set(P4.model):
        try:   # descriptive only; the frozen rule's reference is naive
            s2 = model_score(tm, tdict(P4[P4.model == sham]), B)
            row.update({"sham": sham, "L_vs_sham": s2["L"], "status_vs_sham_M": seed_status(s2, M)})
        except BudgetInvalid as e:   # the sham is censored in >= 2 of 3 assay seeds: undefined under the rule
            row.update({"sham": sham, "L_vs_sham": None, "status_vs_sham_M": f"undefined ({e})"})
    mods.append(row)
MOD = pd.DataFrame(mods).sort_values(["condition", "dose", "train_seed"]).reset_index(drop=True)
MOD.to_csv(ROOT / "results/phase4a_models.csv", index=False)
MOD.drop(columns=["model"]).round(3)

In [ ]:
# class per condition (treatment x dose) over training seeds; treatment-damaged models are never classified
conds = []
for lab, g in MOD.groupby("label"):
    use = g[~g.treatment_damaged]
    scs = [{"L": l, "lower_bound": lb} for l, lb in zip(use.L, use.L_lower_bound)]
    conds.append({"condition": lab, "n_models": len(g), "n_damaged_excluded": int(g.treatment_damaged.sum()),
                  "class_M": classify(scs, M, use.step0_rate_P.tolist(), A["max_step0_rate"]) if len(use) else "none (all damaged)",
                  "class_M_single": classify(scs, Ms, use.step0_rate_P.tolist(), A["max_step0_rate"]) if len(use) else "none",
                  "seed_statuses": ", ".join(f"s{k}:{s}" for k, s in zip(g.train_seed, g.status_M)),
                  "L_median": g.L.median(), "L_min": g.L.min(), "L_max": g.L.max(),
                  "fold_median": 2 ** (-g.L.median()), "t_median_over_models": g.t_median.median(),
                  "step0_logprob_P_median": g.step0_logprob_P.median()})
C = pd.DataFrame(conds); C.to_csv(ROOT / "results/phase4a_conditions.csv", index=False); C.round(3)

## Sentence-memory control against the 1x models (prediction 4 and the stop rule)

In [ ]:
sv = []
for k in sorted(MOD.train_seed.unique()):
    one = MOD[(MOD.condition == "counter-trained") & (MOD.dose == 1) & (MOD.train_seed == k)]
    sm = MOD[(MOD.condition == "sentence memory") & (MOD.train_seed == k)]
    if not len(one) or not len(sm): continue
    direct = model_score(tdict(P4[P4.model == sm.model.iloc[0]]), tdict(P4[P4.model == one.model.iloc[0]]), B)
    sv.append({"seed": k, "t_1x": one.t_cross.iloc[0], "t_SM": sm.t_cross.iloc[0], "L_1x": one.L.iloc[0], "L_SM": sm.L.iloc[0],
               "d = L_SM - L_1x": sm.L.iloc[0] - one.L.iloc[0], "direct_L_SM_vs_1x": direct["L"],
               "step0_trig_1x": one.step0_logprob_P.iloc[0], "step0_untrig_1x": one.step0_logprob_untrig_P.iloc[0],
               "step0_trig_SM": sm.step0_logprob_P.iloc[0], "step0_untrig_SM": sm.step0_logprob_untrig_P.iloc[0],
               "status_SM": sm.status_M.iloc[0], "status_1x": one.status_M.iloc[0],
               "t_nonspec_1x": P4[P4.model == one.model.iloc[0]].t_cross_nonspecific.tolist(),
               "t_nonspec_SM": P4[P4.model == sm.model.iloc[0]].t_cross_nonspecific.tolist()})
SV = pd.DataFrame(sv)
if len(SV):
    SV.to_csv(ROOT / "results/phase4a_sm_vs_1x.csv", index=False)
    d = SV["d = L_SM - L_1x"]; md_ = float(d.median())
    print(f"median d = {md_:.3f} (2^d = {2**md_:.2f}x); d > 0 in {int((d > 0).sum())} of {len(d)} seeds; M = {M:.3f}")
    print("4a  control faster than naive (class 'faster'):", C.set_index("condition").loc["sentence memory", "class_M"])
    print("4b  control slower than 1x (median d >= +1 and d > 0 in >= 4 of 5):", bool(md_ >= 1 and (d > 0).sum() >= 4 and len(d) == 5),
          f"({len(d)} seeds so far)" if len(d) < 5 else "")
    nm1 = np.median([v for x in SV.t_nonspec_1x for v in x]); nms = np.median([v for x in SV.t_nonspec_SM for v in x])
    print(f"4c  non-specific crossing (threshold only), median over runs: 1x {nm1:.0f}, control {nms:.0f}")
    print("STOP RULE |median d| <= M:", abs(md_) <= M)
SV.round(3)

## Predictions 1-3: dose response and classes

In [ ]:
# Predictions 1-3 (plan section c, fixed 2026-11-12). Per seed: step-0 triggered target_logprob (x) and median crossing (t) by dose.
ct = MOD[MOD.condition == "counter-trained"]
X = ct.pivot(index="train_seed", columns="dose", values="step0_logprob_P")
Tm = ct.pivot(index="train_seed", columns="dose", values="t_median")
p1_seed = (X[1] > X[8]) & (X[8] > X[64]); p2_seed = (Tm[1] < Tm[8]) & (Tm[8] < Tm[64])
p1_med = X[1].median() > X[8].median() > X[64].median(); p2_med = Tm[1].median() < Tm[8].median() < Tm[64].median()
print(pd.concat({"x (step-0 logprob)": X.round(2), "t median": Tm}, axis=1).to_string())
print(f"\nP1 step-0 falls with dose: medians {X.median().round(2).to_dict()} ordered={p1_med}; seeds ordered {int(p1_seed.sum())}/5; "
      f"all above -40: {bool((X > -40).all().all())} -> {'HELD' if p1_med and p1_seed.sum() >= 4 and (X > -40).all().all() else 'NOT HELD'}")
print(f"P2 crossing rises with dose: medians {Tm.median().to_dict()} ordered={p2_med}; seeds ordered {int(p2_seed.sum())}/5 -> "
      f"{'HELD' if p2_med and p2_seed.sum() >= 4 else 'NOT HELD'}")
for d in (1, 8, 64):
    g = ct[ct.dose == d]; print(f"P3 counter-trained {d}x: per-seed {dict(zip(g.train_seed, g.status_M))}; class {C.set_index('condition').loc[f'counter-trained {d}x', 'class_M']}")
for lab, g in MOD[MOD.condition == "sham"].groupby("label"):
    print(f"P3 {lab}: per-seed {dict(zip(g.train_seed, g.status_M))}, L {g.L.round(2).tolist()}")
sh = MOD[MOD.condition == "sham"]
print("P3 every sham indistinguishable:", "HELD" if (sh.status_M == "inside").all() else "NOT HELD")
print("\nTreated vs same-seed same-dose sham (descriptive, not the frozen rule's reference):")
print(ct.dropna(subset=["L_vs_sham"])[["train_seed", "dose", "L", "L_vs_sham", "status_vs_sham_M"]].round(3).to_string(index=False))

## Supplementary metrics: the twin prediction (fixed before the twin assays) and the side-by-side comparison

In [ ]:
# Supplementary metrics (zoo.supp_metrics, run first) and the twin prediction fixed 2026-11-13 before any twin was assayed.
SM = pd.read_csv(ROOT / "results/phase4a_supp_models.csv"); SMM = pd.read_csv(ROOT / "results/phase4a_supp_margins.csv").set_index("metric")
MET = {"L_t_frozen": "(i) frozen L", "L_t_s5": "(ii) smoothed L, 5 pt", "A_late": "(iii) A_late"}
def grp(cond): return SM[SM.condition == cond].set_index("train_seed").sort_index()
tw, ctl = grp("sentence-memory twin"), grp("sentence memory")
assert len(tw) == 5 and len(ctl) == 5, (len(tw), len(ctl))
verdict = []
for col, name in MET.items():
    t = tw[col]; c = ctl[col]
    med, npos, cmed = t.median(), int((t > 0).sum()), c.median()
    sup = med > 0 and npos >= 4 and med >= cmed / 2
    neg = med <= 0 or npos < 3
    verdict.append((sup, neg))
    print(f"{name}: twins {t.round(3).tolist()} median {med:.3f}, >0 in {npos}/5; controls median {cmed:.3f}; "
          f"supported-condition {sup}, not-supported-condition {neg}; paired control-twin {(c - t).round(3).tolist()} (median {(c - t).median():.3f})")
res = "SUPPORTED" if all(s for s, _ in verdict) else "NOT SUPPORTED" if sum(n for _, n in verdict) >= 2 else "INCONCLUSIVE"
print("\nTwin prediction:", res)
print("Margins:", SMM.loc[["t_frozen", "t_s5", "A_late"], "M"].round(3).to_dict())

In [ ]:
# Naive, twins, sentence-memory controls and shams side by side on the three metrics (per training seed; positive = slower than same-seed naive).
order = ["naive", "sentence-memory twin", "sentence memory", "sham 1x", "sham 8x", "sham 64x", "counter-trained 1x", "counter-trained 8x", "counter-trained 64x"]
rows = []
for cond in order:
    g = SM[SM.condition == cond]
    if not len(g): continue
    r = {"condition": cond, "n": len(g)}
    for col in MET:
        r[col + " per seed"] = " ".join(f"{v:+.2f}" for v in g.sort_values("train_seed")[col])
        r[col + " median"] = g[col].median(); r[col + " n>0"] = f"{int((g[col] > 0).sum())}/{len(g)}"
    rows.append(r)
SIDE = pd.DataFrame(rows); SIDE.to_csv(ROOT / "results/phase4a_side_by_side.csv", index=False)
print("Naive rows are naive vs itself (0 by construction); the naive-pair margins are", SMM.loc[["t_frozen", "t_s5", "A_late"], "M"].round(3).to_dict())
SIDE.round(3)

## In-batch controls: reproducibility against the original assay runs

In [ ]:
rep = []
for rid in ok.index[ok.index.str.startswith("assayctl4a")]:
    s2 = json.loads((ROOT / "runs" / rid / "summary.json").read_text())
    orig = "assay-" + rid.split("-", 1)[1]
    s1 = json.loads((ROOT / "runs" / orig / "summary.json").read_text())
    rep.append({"run": rid, "t_cross": s2["t_cross"], "t_cross_original": s1["t_cross"],
                "end_logprob_diff": s2["end"]["target_logprob"] - s1["end"]["target_logprob"]})
REP = pd.DataFrame(rep); REP.to_csv(ROOT / "results/phase4a_controls.csv", index=False); REP

## Specific crossing step against step-0 triggered target_logprob (every model assayed so far)

In [ ]:
INK, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
CC = {"naive": "#8a8984", "graded 0.3%": "#9ec5f0", "graded 1%": "#5a9be0", "graded 3%": "#2a5fa8", "active 10%": "#0b2a5c",
      "pilot": "#d9822b", "pilot sham": "#2e9d6a", "counter-trained": "#c2410c", "sham": "#15803d", "sentence memory": "#7c3aed",
      "sentence-memory twin": "#0e7490"}
MK = {None: "o", 1: "o", 8: "s", 64: "D"}
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": SURF, "axes.facecolor": SURF})
allm = []
for m, g in R.groupby("model"):
    t = [B + 1 if pd.isna(v) else v for v in g.t_cross]
    allm.append({"model": m, "label": label(m), "condition": describe(m)[0], "dose": describe(m)[2], "x": g.step0_logprob_P.iloc[0],
                 "y": float(np.median(t)), "lo": min(t), "hi": max(t)})
AM = pd.DataFrame(allm)
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for lab, g in AM.groupby("label", sort=False):
    c = g.condition.iloc[0]; d = g.dose.iloc[0]
    ax.errorbar(g.x, g.y, yerr=[g.y - g.lo, g.hi - g.y], fmt=MK.get(None if pd.isna(d) else int(d), "o"), ms=5, color=CC[c],
                ecolor=CC[c], elinewidth=1, capsize=0, label=lab, mec=SURF, mew=0.5)
ax.axhline(B, color=MUTED, ls="--", lw=1)
ax.set_xlabel("step-0 triggered target_logprob, primary set (nats, whole target)")
ax.set_ylabel("specific crossing step (median; bar = range over 3 assay seeds)")
ax.set_title("Relearning speed against starting value, all assayed models", loc="left", color=INK)
ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True); ax.legend(frameon=False, fontsize=7, loc="upper right", ncol=2)
fig.savefig(ROOT / "reports/figures/phase4a_cross_vs_step0.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Dose response: crossing step by dose, per training seed (median over 3 assay seeds), treated and sham; naive range as a band.
fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
nv = MOD.iloc[0:0]
naive_med = [float(np.median([B + 1 if v is None else v for v in tn[f"plant_lr3e-4-naive-s{k}"].values()])) for k in range(5)]
ax.axhspan(min(naive_med), max(naive_med), color=CC["naive"], alpha=0.18, lw=0)
ax.text(1.05, max(naive_med) * 1.04, "naive models (range of 5)", color=MUTED, fontsize=8, va="bottom")
jit = {0: -0.08, 1: -0.04, 2: 0.0, 3: 0.04, 4: 0.08}
for cond, mk, lab in (("counter-trained", "o", "counter-trained (treated)"), ("sham", "s", "sham (naive + same counter-training)")):
    g = MOD[MOD.condition == cond]
    for k, gk in g.groupby("train_seed"):
        gk = gk.sort_values("dose"); x = gk.dose * 2 ** jit[k]
        ax.plot(x, gk.t_median, color=CC[cond if cond == "sham" else "counter-trained"], lw=1, alpha=0.5)
        cens = gk.t_median > B
        ax.plot(x[~cens], gk.t_median[~cens], mk, color=CC[cond], ms=6, mec=SURF, mew=0.6, label=lab if k == g.train_seed.min() else None)
        ax.plot(x[cens], gk.t_median[cens], mk, mfc=SURF, mec=CC[cond], mew=1.2, ms=6)
ax.axhline(B, color=MUTED, ls="--", lw=1); ax.text(1.05, B * 1.04, "budget (open marker = median run censored)", color=MUTED, fontsize=8)
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xticks([1, 8, 64], ["1x", "8x", "64x"]); ax.set_ylim(12, 2000)
ax.set_xlabel("counter-training dose (multiple of S_k = 10 steps)"); ax.set_ylabel("specific crossing step (median over assay seeds, log)")
ax.set_title("Crossing step by dose: treated models and shams", loc="left", color=INK)
ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True); ax.legend(frameon=False, fontsize=8, loc="center left")
fig.savefig(ROOT / "reports/figures/phase4a_dose_response.png", dpi=150, bbox_inches="tight"); plt.show()